# 2.13 在线学习 (Online Learning)模型在持续到来的数据流上逐步学习更新。本节涵盖：- 增量学习- 流式训练- 经验回放

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ffrom collections import dequeimport randomtorch.manual_seed(42)class OnlineLearner:    def __init__(self, model, lr=1e-3, replay_size=100):        self.model = model        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)        self.replay_buffer = deque(maxlen=replay_size)        def learn_step(self, X, y, use_replay=True, replay_ratio=0.5):        loss = F.cross_entropy(self.model(X), y)                if use_replay and len(self.replay_buffer) > 0:            n_replay = max(1, int(len(X) * replay_ratio))            replay_batch = random.sample(list(self.replay_buffer), min(n_replay, len(self.replay_buffer)))            X_replay = torch.cat([b[0] for b in replay_batch])            y_replay = torch.cat([b[1] for b in replay_batch])            replay_loss = F.cross_entropy(self.model(X_replay), y_replay)            loss = loss + replay_loss                self.optimizer.zero_grad()        loss.backward()        self.optimizer.step()                self.replay_buffer.append((X, y))        return loss.item()model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))learner = OnlineLearner(model, replay_size=50)print('=== Online Learning with Experience Replay ===')n_streams = 20for stream_idx in range(n_streams):    X_stream = torch.randn(8, 10) + stream_idx * 0.3    y_stream = torch.randint(0, 3, (8,))    loss = learner.learn_step(X_stream, y_stream, use_replay=True)        if (stream_idx + 1) % 5 == 0:        X_test = torch.randn(32, 10)        y_test = torch.randint(0, 3, (32,))        acc = (model(X_test).argmax(-1) == y_test).float().mean()        print(f'Stream {stream_idx+1}: loss={loss:.4f}, test_acc={acc:.3f}, buffer_size={len(learner.replay_buffer)}')print('\n=== Without Replay (Catastrophic Forgetting) ===')model_no_replay = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))learner_no = OnlineLearner(model_no_replay, replay_size=0)for stream_idx in range(n_streams):    X_stream = torch.randn(8, 10) + stream_idx * 0.3    y_stream = torch.randint(0, 3, (8,))    loss = learner_no.learn_step(X_stream, y_stream, use_replay=False)print('Without replay buffer, model forgets earlier data patterns.')print('Experience replay mitigates catastrophic forgetting in online learning.')